# Laboratorio 1 · Bitácora

**Nombre: Diego Fernando Irreño Torres**                                                 
**Usuario de GitHub: diegoirreno01-dev**  
**Fecha: 28/08/2026**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla que hace que esto sirva de algo:** la predicción se escribe
> *antes* de ejecutar. Si la rellenas después ya sabiendo el resultado, el
> ejercicio no mide nada y tú no aprendes nada. Nadie va a comprobarlo:
> es un trato contigo mismo.


In [11]:
from rlrs.dp import greedy_policy, q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld
from rlrs.evaluation import compare, evaluate
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

# Este cuaderno es una bitácora: no define algoritmos, los usa.
# Si necesitas escribir una función que valga la pena conservar,
# va en src/rlrs/, no aquí.


## Mi variante

Ejecuta `uv run python scripts/variante.py` y anota lo que te tocó.


In [12]:
RUIDO  = 0.0   # <- rellena
COSTE  = -0.02   # <- rellena
GAMMA  = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)
mi_env.noise, mi_env.step_reward


(0.0, -0.02)

---
## Ejercicio 1 · El respaldo a mano


**Antes de ejecutar.** ¿Cuál de las cuatro acciones crees que gana en (0,3), y por qué?

_Tu predicción:_ Creo que ganaria derecha ya que al realizar el calculo manual me di cuenta que el resultado mas optimo intentado seria llegar a la meta para los 4 movimientos con un valor de 0,93


In [13]:
# tu código
from rlrs.dp import q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld

env = GridWorld()                # el de la guía, no tu variante
valores, politica, barridos = value_iteration(env, gamma=0.9)

for a in range(env.n_actions):
    print(f'{ACTION_NAMES[a]:<10} {q_value(env, valores, (0, 3), a, 0.9):+.6f}')

arriba     +0.800017
derecha    +0.928402
abajo      +0.554337
izquierda  +0.636940


**Explica.** ¿Coincidió? ¿Por qué gana esa y no las otras?

_Tu respuesta:_ Si concidio , hubo una pequeña diferencia por el redondeo de decimales pero lo mas optimo seria la meta para moverse y obtener la recompensa 


---
## Ejercicio 2 · Tu variante, medida


In [14]:
# tu código
from rlrs.evaluation import compare
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)     # los tuyos
valores, politica, barridos = value_iteration(mi_env, gamma=0.9)
print(f'{barridos} barridos · V(3,0) = {valores[mi_env.state_index((3, 0))]:+.4f}')
print(mi_env.render_values(valores, politica))

for r in compare(mi_env,
                 [GreedyTabularPolicy(politica, name='optima'),
                  RandomPolicy(mi_env.n_actions, seed=0)],
                 episodes=300, base_seed=0):
    print(' ', r)

9 barridos · V(3,0) = +0.4377
+0.67>  +0.77>  +0.88>  +1.00>   +1    
+0.59^    ###   +0.77^  +0.88^   -1    
+0.51^  +0.59>  +0.67^    ###   +0.37v 
+0.44^  +0.51^  +0.59^  +0.51<  +0.44< 
  optima         retorno +0.880 [+0.880, +0.880]  exito 100.0%  pasos   7.0
  aleatoria      retorno -1.379 [-1.492, -1.266]  exito 26.0%  pasos  55.9


**Anota.** Barridos, V(3,0), retorno con su intervalo, tasa de éxito y pasos medios.

_Tus números:_ 9 barridos · V(3,0) = +0.4377 , retorno +0.880 [+0.880, +0.880]  exito 100.0%  pasos   7.0


---
## Ejercicio 3 · Subir gamma


**Antes de ejecutar.** Al pasar de 0,9 a 0,99: ¿qué le pasa al número de barridos? ¿Y a la política?

_Tu predicción:_ Aumentarian los barridos porque gamma aumenta el valor de cuanto vale cada casilla por lo que al tener algo de ruido que no es mi caso ya que el ruido es 0 , creo que aumentaria la cantidad de barridos


In [15]:
# tu código
for g in (0.5, 0.9, 0.99):
    v, p, n = value_iteration(mi_env, gamma=g)
    print(f'\ngamma = {g}   {n} barridos   V(3,0) = {v[mi_env.state_index((3, 0))]:+.4f}')
    print(mi_env.render_values(v, p))


gamma = 0.5   9 barridos   V(3,0) = -0.0238
+0.09>  +0.22>  +0.48>  +1.00>   +1    
+0.02^    ###   +0.22^  +0.48^   -1    
-0.01^  +0.02>  +0.09^    ###   -0.03v 
-0.02^  -0.01^  +0.02^  -0.01<  -0.02< 

gamma = 0.9   9 barridos   V(3,0) = +0.4377
+0.67>  +0.77>  +0.88>  +1.00>   +1    
+0.59^    ###   +0.77^  +0.88^   -1    
+0.51^  +0.59>  +0.67^    ###   +0.37v 
+0.44^  +0.51^  +0.59^  +0.51<  +0.44< 

gamma = 0.99   9 barridos   V(3,0) = +0.8244
+0.91>  +0.94>  +0.97>  +1.00>   +1    
+0.88^    ###   +0.94^  +0.97^   -1    
+0.85^  +0.88>  +0.91^    ###   +0.80v 
+0.82^  +0.85^  +0.88^  +0.85<  +0.82< 


**Explica.** ¿Qué se movió mucho y qué se movió poco? ¿Por qué?

_Tu respuesta:_ En mi variante al tener ruido 0 , el camino es determinado a seguir el camino hasta la meta que son 9 casillas por lo que no cambiaran los barridos aunque gamma se altere



---
## Ejercicio 4 · Quitar el ruido


**Antes de ejecutar.** Con ruido 0, ¿cambia la política óptima respecto a la tuya? ¿En qué casillas?

_Tu predicción:_ Mi caso ya tenia ruido 0 pero al no ser asi la politica optima prefiere un camino mas largo pero mas seguro , mientras que al no tener duda de resbalar o cambiar de direccion iria por el camino mas seguro y directo



In [16]:
# tu código
for ruido in (0.0, 0.2, 0.6):
    e = GridWorld(noise=ruido)
    v, p, n = value_iteration(e, gamma=0.9)
    print(f'\nruido = {ruido}   {n} barridos')
    print(e.render_values(v, p))


ruido = 0.0   9 barridos
+0.62>  +0.73>  +0.86>  +1.00>   +1    
+0.52^    ###   +0.73^  +0.86^   -1    
+0.43^  +0.52>  +0.62^    ###   +0.27v 
+0.34^  +0.43^  +0.52^  +0.43<  +0.34< 

ruido = 0.2   35 barridos
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17< 

ruido = 0.6   90 barridos
+0.02^  +0.17>  +0.33>  +0.62>   +1    
-0.08^    ###   +0.21^  +0.28<   -1    
-0.13^  -0.11>  +0.01^    ###   -0.28v 
-0.18^  -0.16>  -0.13<  -0.19<  -0.25v 


**Explica.** ¿Acertaste? Si te sorprendió, di exactamente qué esperabas y qué pasó.

_Tu respuesta:_ Acerte ya habia visto mi ejercicio sin ruido ya que era el valor inicial pero me gusto determinar como se comporto al agregarle ruido y ver su desicion



---
## Ejercicio 5 · Encarecer el paso


**Antes de ejecutar.** Con coste por paso −2, ¿qué hará el agente?

_Tu predicción:_ Tomar desiciones arriesgadas para encontrar la salida mas rapido porque los errores le costaran mas caro 



In [17]:
# tu código
for coste in (-0.001, -0.04, -2.0):
    e = GridWorld(step_reward=coste)
    v, p, n = value_iteration(e, gamma=0.9)
    ev = evaluate(GridWorld(step_reward=coste), GreedyTabularPolicy(p), episodes=300, base_seed=0)
    print(f'coste {coste:>7} · {n:>2} barridos · {ev}')
    print(e.render_values(v, p), '\n')

coste  -0.001 · 41 barridos · avida          retorno +0.991 [+0.991, +0.992]  exito 100.0%  pasos   9.6
+0.62>  +0.71>  +0.82>  +0.94>   +1    
+0.54^    ###   +0.71^  +0.65<   -1    
+0.48^  +0.53>  +0.61^    ###   +0.35v 
+0.42^  +0.47^  +0.52^  +0.46<  +0.40<  

coste   -0.04 · 35 barridos · avida          retorno +0.636 [+0.603, +0.670]  exito 98.0%  pasos   9.1
+0.48>  +0.61>  +0.75>  +0.93>   +1    
+0.37^    ###   +0.61^  +0.59^   -1    
+0.28^  +0.36>  +0.47^    ###   +0.10v 
+0.21^  +0.27^  +0.35^  +0.26<  +0.17<  

coste    -2.0 · 29 barridos · avida          retorno -13.393 [-13.744, -13.043]  exito  6.0%  pasos   7.3
-6.54>  -4.47>  -2.31>  +0.31>   +1    
-8.18^    ###   -3.66>  -1.29>   -1    
-9.20>  -7.71>  -5.86^    ###   -1.46^ 
-10.11>  -8.85>  -7.44>  -5.90>  -3.94^  



**Explica.** ¿Qué está optimizando exactamente el agente para comportarse así?

_Tu respuesta:_ Esta optimizando el terminar antes sin importar el resultado que el llegar de manera mas rapida a la meta


---
## Ejercicio 6 · El error plantado


In [18]:
# ejecuta experiments/divergencia.py desde la terminal y pega aquí lo que salga

#1) value_iteration(env, gamma=1.0)
     #ValueError: gamma debe estar en [0, 1); se recibio 1.0
     #La guardia protege una garantia: sin gamma < 1 el operador de
     #Bellman deja de ser una contraccion.

  #2) gamma = 1.0, recompensa por paso -0.04  (el entorno de siempre)
   #barrido       V(3,0)       max|V|       cambio
  #------------------------------------------------
        # 1      -0.0400       0.7920     0.792000
        #5      -0.2000       0.9513     0.337498
        #10       0.4938       0.9685     0.158584
        #50       0.6675       0.9721     0.000000
       #100       0.6675       0.9721     0.000000
       #500       0.6675       0.9721     0.000000
      #1000       0.6675       0.9721     0.000000
      #2000       0.6675       0.9721     0.000000
     #Converge. Quedarse dando vueltas cuesta -0.04 por paso, o sea
     #-infinito, asi que ninguna politica que el max prefiera lo hace.
     #Es un camino mas corto estocastico, y ahi gamma = 1 esta bien
     #definido. Perder la garantia no es perder la convergencia.

  #3) gamma = 1.0, recompensa por paso +0.01  (ahora le pagamos por moverse)
  # barrido       V(3,0)       max|V|       cambio
  #------------------------------------------------
         #1       0.0100       0.8020     0.802000
         #5       0.0500       0.9703     0.368426
        #10       0.8791       1.0174     0.180948
        #50       1.4173       1.4173     0.010000
       #100       1.9173       1.9173     0.010000
       #500       5.9173       5.9173     0.010000
      #1000      10.9173      10.9173     0.010000
      #2000      20.9173      20.9173     0.010000
     #No converge. A partir del barrido 100 los valores crecen +0.01
     #por barrido, indefinidamente, y el cambio se estanca en 0.01:
     #el criterio de parada nunca se dispara.

  #El mismo entorno con gamma = 0.9 converge en 42 barridos,
  #con max|V| = 0.9496. La unica diferencia es el descuento.

  #El diagnostico tiene dos capas.
    #Matematica: con gamma = 1 existe una politica que nunca termina y
    #acumula +0.01 sin fin, luego su retorno es +infinito. No hay punto
    #fijo finito al que converger.
    #De diseno: el error no fue poner gamma = 1, fue pagar por el
    #proceso en vez de por el resultado. El descuento solo lo tapaba.


**Explica las dos capas del diagnóstico.**

_La capa matemática:_ Sin descuento, el operador de Bellman deja de ser una contracción, así que ya no hay garantía de un único punto fijo. Eso no siempre causa divergencia: con coste −0,04 todas las políticas terminan (dar vueltas cuesta −infinito), así que el problema sigue bien definido y converge. Pero con recompensa +0,01 por paso existe una política que nunca termina y acumula +infinito, así que no hay ningún valor finito al que converger — por eso el cambio nunca baja de 0,01 y los valores crecen sin parar.

_La capa de diseño:_ El problema real no fue usar gamma=1, sino premiar el proceso (moverse) en vez del resultado (llegar a la meta). Con recompensa positiva por paso, la política óptima matemáticamente correcta es no terminar nunca. El descuento no corregía ese error, solo lo ocultaba al ponerle un tope al valor de dar vueltas para siempre.


---
## Cierre

**Lo que más me sorprendió hoy:** Como se comporta el agente frente al entorno para tomar sus decisiones y el procesamiento que hace para determinarlo

**Lo que todavía no entiendo:** Aun no comprendo muy bine la terminologia de las variables dentro de las formulas pero viendolas corriendo en codigo me permite comprenderlas un poco mejor
